# 6.1 — NumPy Arrays and RDDs

**Chapter 6, section 6.1**, and the starting point for **Exercises 1 and 2**.

**The question this notebook answers:** a PySpark program has two numerical layers, NumPy inside
one task and Spark across many, and the last cell of this notebook is where they meet. Everything
before it is the recap that makes that cell readable: how an array is created, what an arithmetic
operator does to one, and why an elementwise product is not yet a dot product.

The payload is the final expression,

$$\theta = \sum_{i=1}^{n} x_i \odot \beta ,$$

computed as `rdd.map(lambda x: x * beta).reduce(lambda x, y: x + y)`. The `map` is the
elementwise part and runs where each record lives; the `reduce` is the summation and is the only
communication. That split is the shape of every distributed gradient computation in this course.

Source: `Notebooks old/Spark-Example-08-PySpark and NumPy.ipynb`, modernized and revised.
Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
import numpy as np
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-6.1")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark", spark.version, "| NumPy", np.__version__, "| cores", sc.defaultParallelism)

Spark 4.2.0 | NumPy 2.5.2 | cores 4


---

## 1. Creating an array

Four calls cover most arrays anyone creates by hand.

In [2]:
# To create a NumPy array, you can use the function np.array()
a = np.array([2, 1, 2, 1])
print(a)

# Create an array filled with 0's
b = np.zeros(3)
print(b)

# Array filled with 1's
c = np.ones(3)
print(c)

# Array with a range of elements
d = np.arange(4)
print(d)

[2 1 2 1]
[0. 0. 0.]
[1. 1. 1.]
[0 1 2 3]


The decimal points in the second and third outputs are not cosmetic. `np.array` infers an integer
type from integer literals, while `np.zeros` and `np.ones` default to floating point. The `dtype`
silently governs every later operation, so a wrong inference at creation surfaces far from its
cause: checking `a.dtype` is the numerical analogue of checking a DataFrame's schema.

In [3]:
for name, arr in [("a", a), ("b", b), ("d", d)]:
    print(f"{name}: dtype={arr.dtype}  shape={arr.shape}  ndim={arr.ndim}  size={arr.size}")

a: dtype=int64  shape=(4,)  ndim=1  size=4
b: dtype=float64  shape=(3,)  ndim=1  size=3
d: dtype=int64  shape=(4,)  ndim=1  size=4


---

## 2. Whole-array arithmetic

Operators applied to two arrays act **elementwise**, pairing corresponding entries. The loop over
elements happens in compiled code, not in the interpreter; that is what *vectorized* means.

In [4]:
print("a     =", a)
print("d     =", d)
print("a + d =", a + d)
print("a - d =", a - d)
print("a * d =", a * d)     # elementwise, NOT a dot product -- see below
print("a.sum() =", a.sum())
print("a.min() =", a.min())

a     = [2 1 2 1]
d     = [0 1 2 3]
a + d = [2 2 4 4]
a - d = [ 2  0  0 -2]
a * d = [0 1 4 3]
a.sum() = 6
a.min() = 1


### The one reading error worth forestalling

`a * d` is an **elementwise product**, sometimes called the Hadamard product. A **dot product** is
an elementwise product *followed by a reduction*, $\sum_i a_i d_i$.

The distinction looks pedantic on four-element vectors. It is the architecture of distributed
model fitting: the elementwise part can happen independently on each record wherever it lives,
while the summation is a communication step that brings partial results together. Section 5 of
this notebook is that same pair, spread over a cluster.

(The original course notebook labelled `a * d` "dot product of the vectors". It is not, and the
cell below is the difference.)

In [5]:
print("a * d          =", a * d)           # elementwise: an array
print("(a * d).sum()  =", (a * d).sum())   # reduced:     a scalar
print("np.dot(a, d)   =", np.dot(a, d))    # the same scalar, in one call

assert (a * d).sum() == np.dot(a, d)

a * d          = [0 1 4 3]
(a * d).sum()  = 8
np.dot(a, d)   = 8


---

## 3. Broadcasting

Combining an array with a single number works, and the mechanism behind it is **broadcasting**:
NumPy's rule for applying an operation between arrays of different shapes by conceptually
stretching the smaller across the larger, without copying it.

In [6]:
print("a * 5 =", a * 5)
print("a + 3 =", a + 3)
print("a / 2 =", a / 2)

a * 5 = [10  5 10  5]
a + 3 = [5 4 5 4]
a / 2 = [1.  0.5 1.  0.5]


The rule is checked shape by shape **from the trailing dimension leftward**: two dimensions are
compatible when they are equal or when one of them is 1, and a missing dimension counts as 1. So a
vector of shape `(3,)` stretches down the four rows of a `(4, 3)` matrix.

In [7]:
m = np.array([[0, 0, 0], [10, 10, 10], [20, 20, 20], [30, 30, 30]])
v = np.array([1, 2, 3])

print("m.shape =", m.shape, " v.shape =", v.shape)
print(m + v)

m.shape = (4, 3)  v.shape = (3,)
[[ 1  2  3]
 [11 12 13]
 [21 22 23]
 [31 32 33]]


A pairing whose trailing dimensions are neither equal nor 1 is an error rather than a guess. The
cell below provokes it on purpose, because the error message is the thing to recognize.

In [8]:
bad = np.array([1, 2, 3, 4])          # shape (4,) against a (4, 3) matrix
try:
    m + bad
except ValueError as e:
    print("ValueError:", str(e).splitlines()[0])

ValueError: operands could not be broadcast together with shapes (4,3) (4,) 


Two warnings travel with broadcasting. First, it is a claim about how a loop is *executed*, not a
guarantee that the loop's output is small: a broadcast between high-dimensional arrays can imply a
very large intermediate result. Second, the word collides. **NumPy broadcasting** is a shape rule
inside one process; a **Spark broadcast variable** is a read-only value shipped once to each
executor; a **broadcast join** copies the small side of a join to every executor. This notebook
means the first.

---

## 4. A formula as one expression

The payoff of vectorization is that a mathematical formula becomes a single line. Take the mean
squared error,

$$\mathrm{error} = \frac{1}{n}\sum_{i=1}^{n}\left(\mathrm{label}_i-\mathrm{prediction}_i\right)^2 .$$

In [9]:
predictions = np.array([2.2, 3.3, 3.9, 5.1, 5.3])
labels = np.array([2, 3, 4, 5, 6])
n = len(predictions)

error = (1 / n) * np.sum(np.square(labels - predictions))
print(error)

0.12800000000000003


The expression reads exactly as the formula does: the subtraction is elementwise, `np.square` is
elementwise, `np.sum` is the reduction, and the division by $n$ is a scalar broadcast. No Python
loop appears and none is hidden. This is, cell for cell, the error computation inside a gradient
descent implementation of regression.

---

## 5. NumPy inside an RDD

Nothing about an RDD restricts what a record may be, and a record may perfectly well be a NumPy
array. Handed a two-dimensional array, `parallelize` splits it along its first axis, so each
record of the resulting RDD is **one row**.

In [10]:
a2 = np.array([[1, 2, 3], [4, 5, 6]])
rdd = sc.parallelize(a2)
print(rdd.collect())

[array([1, 2, 3]), array([4, 5, 6])]


That output is the fact to notice: not one $2\times3$ array but a **list of two one-dimensional
arrays**, because the matrix has been cut into row records and distributed.

The `map` below is executed on the executors. The lambda is serialized and shipped with each task,
and inside each task `x * 2` is an ordinary NumPy scalar broadcast over one row. The driver never
sees the intermediate rows; `collect()` is used here, as throughout these small examples, only
because the data is tiny enough to inspect.

In [11]:
print(rdd.map(lambda x: x * 2).collect())

[array([2, 4, 6]), array([ 8, 10, 12])]


### A record is usually a pair

Machine learning data rarely travels as bare vectors: a record pairs an identifier or a label with
its features, and the natural RDD shape for that is a key–value pair whose value is an array.

In [12]:
a6 = np.arange(30).reshape(6, 5)
keys = np.arange(6)
rdd = sc.parallelize(list(zip(keys, a6)))

params = np.array([0.5, 2, 0.5, 2, 0.5])
for k, row in rdd.map(lambda x: (x[0], x[1] * params)).collect():
    print(int(k), row)

0 [0. 2. 1. 6. 2.]
1 [ 2.5 12.   3.5 16.   4.5]
2 [ 5. 22.  6. 26.  7.]
3 [ 7.5 32.   8.5 36.   9.5]
4 [10. 42. 11. 46. 12.]
5 [12.5 52.  13.5 56.  14.5]


Two things happen in that `map`, one numerical and one distributed.

**Numerically**, `x[1] * params` is again the elementwise product, weighting each of the five
features of a row by the corresponding parameter. As before, it is not a dot product until it is
summed.

**Distributively**, `params` is a plain variable referred to inside the lambda, so it is captured
in the function's closure and shipped with **every task**. A value used this way by many tasks is
better sent as a broadcast variable, transmitted once per executor. For a five-element array the
difference is negligible; for a large model it is not.

The pattern of this cell — a fixed parameter vector applied identically to every record of a
distributed dataset — is the inner loop of every distributed model fit.

---

## 6. The distributed sum

This is the cell the whole notebook has been arranging. It computes

$$\theta \;=\; \sum_{i=1}^{n} x_i \odot \beta ,$$

where $x_1,\dots,x_n$ are the rows of a matrix and $\beta$ is a parameter vector, so that $\theta$
is a five-element vector whose $j$th entry is $\beta_j\sum_i x_{ij}$.

In [13]:
x = np.arange(30).reshape(6, 5)
rdd = sc.parallelize(x)
beta = np.array([0.5, 1, 0.3, 0.2, 2])

theta = rdd.map(lambda x: x * beta).reduce(lambda x, y: x + y)
print(theta)

[ 37.5  81.   26.1  18.6 198. ]


The structure deserves more attention than the arithmetic.

* The **`map` is narrow**: each record is weighted by $\beta$ locally, inside its own task, with no
  communication.
* The **`reduce` is the communication step**. NumPy's `+` adds vectors elementwise, and because
  vector addition is associative, Spark is free to add partial sums within each partition first and
  then combine the per-partition results. What crosses the network is one five-element vector per
  partition rather than one per record, and the final sum arrives at the driver as a single small
  array.

Both halves are checkable, and a claim that runs is worth more than a claim that is asserted. The
same quantity computed by NumPy alone must agree exactly.

In [14]:
local = (x * beta).sum(axis=0)      # the same sum, entirely in one process
print("distributed :", theta)
print("local       :", local)
assert np.allclose(theta, local)

# and the per-partition structure the reduce is exploiting
# what one executor contributes: the sum of its own weighted rows
def partial_sum(rows):
    rows = np.array(list(rows))
    return np.zeros(len(beta)) if rows.size == 0 else (rows * beta).sum(axis=0)

# three partitions, fixed, so this printout is the same on every machine
rdd3 = sc.parallelize(x, 3)
partials = rdd3.glom().map(partial_sum).collect()
print("partitions   :", rdd3.getNumPartitions())
for i, p in enumerate(partials):
    print(f"  partition {i}:", p)
print("their sum    :", sum(partials))

distributed : [ 37.5  81.   26.1  18.6 198. ]
local       : [ 37.5  81.   26.1  18.6 198. ]
partitions   : 3
  partition 0: [ 2.5  7.   2.7  2.2 26. ]
  partition 1: [12.5 27.   8.7  6.2 66. ]
  partition 2: [ 22.5  47.   14.7  10.2 106. ]
their sum    : [ 37.5  81.   26.1  18.6 198. ]


Each partition contributes one five-element vector; the driver adds those together. When the
vector is large — a model with a million parameters rather than five — `reduce` sends one such
vector per partition to the driver at once, and the right call is `treeAggregate`, which combines
partial results in a tree of intermediate stages on the executors instead.

---

## Conclusion

A NumPy array inside a `map` is a **per-record object**: thousands of small arrays exist
simultaneously across the cluster, one inside each task, and no NumPy operation ever sees more
than the records of its own partition. The only way out of the distributed world is an action such
as `collect()`, and a program that takes it has silently accepted a bound of one machine's memory.

The elementwise product and the reduction are separable, and separating them is what makes model
fitting distributable: **a per-record term computed where the record lives, folded into a
per-partition partial, and combined associatively into one vector at the driver.**

Exercises 1 and 2 of the chapter start here: turn the elementwise product into a true per-record
dot product, then build a distributed mean squared error out of `map` and `reduce` alone.

In [15]:
spark.stop()